# exp_e2_mpnet_multi — Semantic Graph Builder v2

**Phase 1a, embedder = `paraphrase-multilingual-mpnet-base-v2`, LLM = `deepseek-v32/latest` (API).**

Запускается из `exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/` — все пути разрешаются автоматически от `clustering_1/`.

Перед запуском: `export YANDEX_CLOUD_API_KEY=...`.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch

In [2]:
import os, sys, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# layout: clustering_1/exps/FINAL_EXPS/phase1a_embedders/exp_e2_mpnet_multi/
EXP_DIR   = Path().resolve()
REPO_ROOT = EXP_DIR.parents[3]                 # clustering_1/
LLM_V2    = REPO_ROOT / 'llm_v2'

# put repo root on sys.path so `import llm_v2` works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# guard: never expose llm_v2/ as flat path
if str(LLM_V2) in sys.path:
    sys.path.remove(str(LLM_V2))

from llm_v2.config_schema import load_config
config = load_config(EXP_DIR / 'config.yaml')

# expand ${YANDEX_CLOUD_API_KEY} etc. (no-op for local LLMs)
config.llm.api_key = os.path.expandvars(config.llm.api_key)
config.llm.base_url = os.path.expandvars(config.llm.base_url)
config.llm.folder = os.path.expandvars(config.llm.folder)

print('EXP_DIR  :', EXP_DIR)
print('REPO_ROOT:', REPO_ROOT)
print('LLM_V2   :', LLM_V2)
print()
print(config.model_dump_json(indent=2))

EXP_DIR  : /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l02_qwen_7b
REPO_ROOT: /home/platoon/graph/semantic-graph
LLM_V2   : /home/platoon/graph/semantic-graph/llm_v2

{
  "llm": {
    "provider": "local",
    "model_name": "Qwen/Qwen2.5-7B-Instruct",
    "max_new_tokens": 500,
    "temperature": 0.3,
    "device": "cuda",
    "load_in_8bit": false,
    "api_key": "",
    "base_url": "",
    "folder": "",
    "instructions": ""
  },
  "embedding": {
    "model_name": "intfloat/multilingual-e5-large",
    "device": "cuda"
  },
  "coreference": {
    "enabled": false,
    "prompt_file": "prompts/coreference_ru.txt",
    "context_sentences": 3,
    "window_sentences": 5
  },
  "extraction": {
    "prompt_file": "prompts/extraction_ru.txt",
    "chunk_size": 3,
    "overlap_size": 1
  },
  "normalization": {
    "enabled": true,
    "language": "ru"
  },
  "deduplication": {
    "enabled": true,
    "threshold": 0.92
  },
  "clustering": {
    "method": "agglomerativ

In [4]:
config.llm.api_key = ""

In [3]:
from llm_v2.models.llm_client import LLMClient
from llm_v2.models.embedder import Embedder

llm = LLMClient(config.llm)
embedder = Embedder(config.embedding)
print(f'LLM loaded: {config.llm.model_name}')
print(f'Embedder loaded: {config.embedding.model_name} (dim={embedder.dim})')

/home/platoon/graph/graph_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-07 16:19:24,926 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 16:19:25,085 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-05-07 16:19:25,246 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-05-07 16:19:25,402 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 16:19:25,564 [

LLM loaded: Qwen/Qwen2.5-7B-Instruct
Embedder loaded: intfloat/multilingual-e5-large (dim=1024)


In [5]:
from llm_v2.utils.io import load_text

input_path = Path(config.paths.input_text)
if not input_path.is_absolute():
    input_path = (LLM_V2 / input_path).resolve()
text = load_text(input_path)
print(f'Input: {input_path}')
print(f'Length: {len(text)} chars')
print(text[:500])

Input: /home/platoon/graph/semantic-graph/benchmark/final_bench/formated_fragment2.md
Length: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.

В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам 


## [0] Preprocessing

In [6]:
from llm_v2.stages.preprocessing import preprocess

sentences = preprocess(text, language=config.normalization.language)
for s in sentences:
    print(f'  [{s.id}] {s.text}')

  [0] # Линейная классификация

Теперь давайте поговорим про задачу классификации.
  [1] Для начала будем говорить про бинарную классификацию на два класса.
  [2] Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.
  [3] Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$.
  [4] В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$.
  [5] Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого.
  [6] **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой.
  [7] Выборка, для которой это возможно, называется линейно разделимой.
  [8] Увы, в реальной жизни такое встречается кр

## [1] Coreference Resolution

In [7]:
from llm_v2.stages.coreference import resolve_coreferences

resolved_text, sentences = resolve_coreferences(
    sentences, llm, config.coreference, base_dir=LLM_V2
)
print('Resolved text:')
print(resolved_text)
print(f'\nSentences after coref: {len(sentences)}')

Resolved text:
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть теперь наши таргеты $y$ кодируют принадлежность к положительному или отрицательному классу, то есть принадлежность множеству $\{-1,1\}$, а $x$ — по-прежнему векторы из $\mathbb{R}^D$. В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$. Мы хотим обучить линейную модель так, чтобы плоскость, которую она задаёт, как можно лучше отделяла объекты одного класса от объектов другого. **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: положительный окажется с одной стороны от неё, а отрицательный — с другой. Выборка, для которой это возможно, называется линейно разделимой. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель

## [1.5] Chunking

In [8]:
from llm_v2.stages.chunking import build_chunks

chunks = build_chunks(sentences, config.extraction)
for c in chunks:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:80]}...')

  chunk_0 (sents [0, 1, 2]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для...
  chunk_1 (sents [2, 3, 4]): Обобщить эту задачу до задачи классификации на $K$ классов не составит большого ...
  chunk_2 (sents [4, 5, 6]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам буду...
  chunk_3 (sents [6, 7, 8]): **2.1.6**

В идеальной ситуации найдётся плоскость, которая разделит классы: пол...
  chunk_4 (sents [8, 9, 10]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модел...
  chunk_5 (sents [10, 11, 12]): $$

<details>
<summary>Почему бы не решать задачу классификации как задачу регре...
  chunk_6 (sents [12, 13, 14]): Во вторых, ошибкой будет считаться предсказание, например, $5$ вместо $1$, хотя ...
  chunk_7 (sents [14, 15, 16]): </details>

Сконструируем теперь функционал ошибки так, чтобы он вышеперечисленн...
  chunk_8 (sents [16, 17, 18]): $$

Домножим обе части на $y_i$ и немного упростим:

$

## [2] Triplet Extraction

In [9]:
from llm_v2.stages.extraction import extract_triplets

raw_triplets = extract_triplets(chunks, llm, config.extraction, base_dir=LLM_V2)
print(f'Extracted {len(raw_triplets)} raw triplets:')
for t in raw_triplets:
    print(f'  {t.subject} | {t.relation} | {t.object}  [{t.chunk_id}]')

Extracting triplets: 100%|██████████| 56/56 [10:46<00:00, 11.54s/it]

Extracted 880 raw triplets:
  задача | является | классификацией  [chunk_0]
  задача классификации | обобщается до | задачи классификации на K классов  [chunk_0]
  задача классификации | включает в себя | бинарную классификацию на два класса  [chunk_0]
  набор данных | нужно разделить на | два класса  [chunk_0]
  набор данных | разделяется с помощью | линейной границы  [chunk_0]
  линейная граница | разделяет | пространство на две части  [chunk_0]
  точка | относится к | одному классу  [chunk_0]
  если точка находится выше | линейной границы, то | она относится к одному классу  [chunk_0]
  если точка находится ниже | линейной границы, то | она относится к другому классу  [chunk_0]
  алгоритм | имеет | свои преимущества и недостатки  [chunk_0]
  логистическая регрессия | хорошо работает при | наличии небольшого количества признаков  [chunk_0]
  логистическая регрессия | может быть чувствительна к | выбросам  [chunk_0]
  случайный лес | может справиться с | большим числом признаков  [chu

## [3] Normalization

In [10]:
from llm_v2.stages.normalization import normalize_triplets

norm_triplets = normalize_triplets(raw_triplets, config.normalization)
print(f'Normalized {len(norm_triplets)} triplets:')
for t in norm_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

2026-05-07 16:31:04,143 [INFO] Loading dictionaries from /home/platoon/graph/graph_env/lib/python3.12/site-packages/pymorphy3_dicts_ru/data


2026-05-07 16:31:04,170 [INFO] format: 2.4, revision: 417150, updated: 2022-01-08T22:09:24.565962


Normalized 880 triplets:
  задача | являться | классификация
  задача классификация | обобщаться до | задача классификация на k класс
  задача классификация | включать в себя | бинарный классификация на два класс
  набор данные | нужно разделить на | два класс
  набор данные | разделяться с помощь | линейный граница
  линейный граница | разделять | пространство на два часть
  точка | относиться к | один класс
  если точка находиться выше | линейный границы, то | она относиться к один класс
  если точка находиться ниже | линейный границы, то | она относиться к другой класс
  алгоритм | иметь | преимущество и недостаток
  логистический регрессия | хорошо работать при | наличие небольшой количество признак
  логистический регрессия | мочь быть чувствительный к | выброс
  случайный лес | мочь справиться с | больший число признак
  случайный лес | требовать большой время для | обучение
  задача | обобщаться до | задача классификация на k класс
  таргет y | кодировать | принадлежность к поло

## [4] Deduplication

In [11]:
from llm_v2.stages.deduplication import deduplicate_triplets

dedup_triplets = deduplicate_triplets(norm_triplets, embedder, config.deduplication)
print(f'After dedup: {len(norm_triplets)} -> {len(dedup_triplets)} triplets')
for t in dedup_triplets:
    print(f'  {t.norm_subject} | {t.norm_relation} | {t.norm_object}')

After dedup: 880 -> 494 triplets
  задача | являться | классификация
  задача классификация | обобщаться до | задача классификация на k класс
  задача классификация | включать в себя | бинарный классификация на два класс
  набор данные | нужно разделить на | два класс
  набор данные | разделяться с помощь | линейный граница
  линейный граница | разделять | пространство на два часть
  точка | относиться к | один класс
  если точка находиться ниже | линейный границы, то | она относиться к другой класс
  алгоритм | иметь | преимущество и недостаток
  логистический регрессия | хорошо работать при | наличие небольшой количество признак
  логистический регрессия | мочь быть чувствительный к | выброс
  случайный лес | мочь справиться с | больший число признак
  случайный лес | требовать большой время для | обучение
  таргет y | кодировать | принадлежность к положительный или отрицательный класс
  x | являться | вектор из r^d
  класс | обозначаться | множество {-1,1}
  метка | встречаться в жи

## [5] Graph Assembly (raw)

In [12]:
from llm_v2.stages.graph_assembly import assemble_graph

raw_graph = assemble_graph(dedup_triplets, chunks, text, config)
print(f'Raw graph: {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nNodes:')
for n in raw_graph.nodes:
    print(f'  {n.id}: {n.label} ({len(n.mentions)} mentions)')
print('\nEdges:')
for e in raw_graph.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (w={e.weight})')

Raw graph: 680 nodes, 494 edges

Nodes:
  n0: задача (3 mentions)
  n1: классификация (1 mentions)
  n2: задача классификация (3 mentions)
  n3: задача классификация на k класс (1 mentions)
  n4: бинарный классификация на два класс (1 mentions)
  n5: набор данные (5 mentions)
  n6: два класс (1 mentions)
  n7: линейный граница (2 mentions)
  n8: пространство на два часть (1 mentions)
  n9: точка (2 mentions)
  n10: один класс (1 mentions)
  n11: если точка находиться ниже (1 mentions)
  n12: она относиться к другой класс (1 mentions)
  n13: алгоритм (4 mentions)
  n14: преимущество и недостаток (1 mentions)
  n15: логистический регрессия (5 mentions)
  n16: наличие небольшой количество признак (1 mentions)
  n17: выброс (1 mentions)
  n18: случайный лес (2 mentions)
  n19: больший число признак (1 mentions)
  n20: обучение (1 mentions)
  n21: таргет y (1 mentions)
  n22: принадлежность к положительный или отрицательный класс (1 mentions)
  n23: x (1 mentions)
  n24: вектор из r^d (1 me

## [6] Clustering

In [13]:
from llm_v2.stages.clustering import cluster_graph, cluster_graph_multi, cluster_graph_all_methods
from llm_v2.utils.io import load_prompt

naming_prompt_path = Path(config.clustering.cluster_naming_prompt)
if not naming_prompt_path.is_absolute():
    naming_prompt_path = (LLM_V2 / naming_prompt_path).resolve()
naming_prompt = load_prompt(naming_prompt_path) if naming_prompt_path.exists() else None

if config.clustering.multi_method:
    multi = cluster_graph_all_methods(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    print('Multi-method clustering:')
    for method_name, mr in multi.methods.items():
        print(f'  {method_name}: {len(mr.param_labels)} variants')
        for lbl in mr.param_labels:
            g = mr.graphs[lbl]
            print(f'    {lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    agg = multi.methods['agglomerative']
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
elif config.clustering.is_multi_threshold:
    multi = cluster_graph_multi(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )
    agg = multi.methods['agglomerative']
    print(f'Multi-threshold: {len(agg.param_labels)} levels')
    for lbl in agg.param_labels:
        g = agg.graphs[lbl]
        print(f'  t={lbl}: {len(g.nodes)} nodes, {len(g.edges)} edges')
    mid_label = agg.param_labels[len(agg.param_labels) // 2]
    clustered = agg.graphs[mid_label]
else:
    clustered = cluster_graph(
        raw_graph, embedder, config,
        llm=llm, prompt_template=naming_prompt,
    )

print(f'\nClustered graph: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print('\nClustered Nodes:')
for n in clustered.nodes:
    print(f'  {n.id}: {n.label} (members={n.members}, size={n.size})')
print('\nClustered Edges:')
for e in clustered.edges:
    print(f'  {e.id}: {e.source} --[{e.label}]--> {e.target} (size={e.size})')

Multi-method clustering:
  agglomerative: 10 variants
    0.250: 197 nodes, 241 edges
    0.322: 79 nodes, 135 edges
    0.394: 55 nodes, 111 edges
    0.467: 33 nodes, 89 edges
    0.539: 15 nodes, 49 edges
    0.611: 3 nodes, 5 edges
    0.683: 1 nodes, 0 edges
    0.756: 1 nodes, 0 edges
    0.828: 1 nodes, 0 edges
    0.900: 1 nodes, 0 edges
  kmeans: 4 variants
    k=10: 10 nodes, 55 edges
    k=25: 25 nodes, 67 edges
    k=40: 40 nodes, 105 edges
    k=55: 55 nodes, 105 edges
  hdbscan: 9 variants
    mcs=3,ms=1: 96 nodes, 151 edges
    mcs=3,ms=3: 107 nodes, 169 edges
    mcs=3,ms=5: 137 nodes, 187 edges
    mcs=5,ms=1: 107 nodes, 153 edges
    mcs=5,ms=3: 121 nodes, 179 edges
    mcs=5,ms=5: 137 nodes, 187 edges
    mcs=10,ms=1: 144 nodes, 165 edges
    mcs=10,ms=3: 154 nodes, 188 edges
    mcs=10,ms=5: 158 nodes, 189 edges

Clustered graph: 3 nodes, 5 edges

Clustered Nodes:
  c0: мы (members=['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10', 'n11', 'n12', 'n1

## Save outputs

In [14]:
from llm_v2.utils.io import save_json, save_text

out = EXP_DIR / config.paths.output_dir
out.mkdir(parents=True, exist_ok=True)

save_text(resolved_text, out / 'coreference_resolved.txt')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')

if config.clustering.multi_method or config.clustering.is_multi_threshold:
    save_json(multi.model_dump(), out / 'multi_clustered_graph.json')
    method_counts = {m: len(r.param_labels) for m, r in multi.methods.items()}
    print(f'Saved multi_clustered_graph.json (methods: {method_counts})')

print(f'Saved to {out}/')

Saved multi_clustered_graph.json (methods: {'agglomerative': 10, 'kmeans': 4, 'hdbscan': 9})
Saved to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l02_qwen_7b/output/


## Benchmark vs ground-truth graph

In [15]:
from llm_v2.benchmark import (
    evaluate_graph,
    evaluate_multi_graph,
    load_clustered_graph,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

# GT inputs (absolute, robust to CWD)
gt_graph_path = REPO_ROOT / 'benchmark' / 'final_bench' / 'graph_clustered.json'
gt_text_path  = REPO_ROOT / 'benchmark' / 'final_bench' / 'formated_fragment2.md'

gt_graph = load_clustered_graph(gt_graph_path)
gt_text  = gt_text_path.read_text(encoding='utf-8')

# embedding context: prefer the coreference-resolved text the pipeline saw
source_text = resolved_text if resolved_text else gt_text

print(f'GT  : {len(gt_graph.nodes)} nodes, {len(gt_graph.edges)} edges  ({gt_graph_path})')
print(f'Pred: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
print(f'Context text: {len(source_text)} chars')

TAU_NODE = 0.6
TAU_EDGE = 0.6
BETA = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

GT  : 55 nodes, 51 edges  (/home/platoon/graph/semantic-graph/benchmark/final_bench/graph_clustered.json)
Pred: 3 nodes, 5 edges
Context text: 15418 chars


In [16]:
metrics = evaluate_graph(
    pred=clustered,
    gt=gt_graph,
    source_text=source_text,
    embedder=embedder,
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

Batches: 100%|██████████| 1/1 [00:00<00:00, 107.68it/s]


GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.1191

Nodes (pred=3, gt=55, matched=3, tau=0.6, beta=1):
  TP(soft)  = 2.6682
  precision = 0.8894
  recall    = 0.0485
  F1        = 0.0920
Edges (pred=5, gt=51, matched=5, tau=0.6, beta=1):
  TP(soft)  = 4.4760
  precision = 0.8952
  recall    = 0.0878
  F1        = 0.1599


{'graph_score': 0.11914723489084858,
 'node_weight': 0.6,
 'edge_weight': 0.4,
 'nodes': {'precision': 0.8894141912460327,
  'recall': 0.0485135013406927,
  'f_beta': 0.09200836461165858,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 2.668242573738098,
  'pred_count': 3,
  'gt_count': 55,
  'matched_count': 3},
 'edges': {'precision': 0.8951910257339477,
  'recall': 0.08776382605234782,
  'f_beta': 0.15985554030963356,
  'beta': 1.0,
  'tau': 0.6,
  'tp': 4.475955128669739,
  'pred_count': 5,
  'gt_count': 51,
  'matched_count': 5},
 'pred_structure': {'n_nodes': 3,
  'n_edges': 5,
  'density': 0.8333333333333334,
  'n_components': 1,
  'n_isolated': 0,
  'component_sizes': [3],
  'component_size_min': 3,
  'component_size_max': 3,
  'component_size_mean': 3.0,
  'component_size_quantiles': {'q25': 3.0,
   'q50': 3.0,
   'q75': 3.0,
   'q90': 3.0}},
 'gt_structure': {'n_nodes': 55,
  'n_edges': 51,
  'density': 0.01717171717171717,
  'n_components': 6,
  'n_isolated': 0,
  'component_sizes': [2

In [17]:
show_node_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched node pairs: 3 / min(3, 55)=3

Top 3 matched (by quality q):
  [q=0.898]  'мы'  ↔  'число ошибок классификатора'
  [q=0.894]  'SVM'  ↔  'вектор'
  [q=0.876]  'природа'  ↔  'минимальный отступ'

Unmatched GT nodes (52):
  'задача классификации'
  'бинарная классификация'
  'классификация на $K$ классов'
  'таргет $y$'
  'положительный класс'
  'отрицательный класс'
  'множество $\\{-1, 1\\}$'
  'признак $x_i$'
  'пространство $\\mathbb{R}^D$'
  'линейная модель'


In [18]:
show_edge_alignments(clustered, gt_graph, metrics, top_k=TOP_K)

Matched edge pairs: 5 / min(5, 51)=5

Top 5 matched (by quality q):
  [q=0.917]  SVM —[быть доступный]→ мы
           ↔  вектор —[принадлежит]→ пространство $\mathbb{R}^D$
  [q=0.901]  природа —[являться]→ мы
           ↔  регрессия —[является плохим подходом для]→ задача классификации
  [q=0.890]  мы —[провести]→ SVM
           ↔  регрессия —[может быть наивным подходом к]→ задача классификации
  [q=0.886]  мы —[работать в]→ природа
           ↔  выборка —[может обладать свойством]→ линейная разделимость
  [q=0.882]  SVM —[лежать на]→ природа
           ↔  признак $x_i$ —[является]→ вектор

Unmatched GT edges (46):
  бинарная классификация —[является частным случаем]→ задача классификации
  бинарная классификация —[обобщается до]→ классификация на $K$ классов
  таргет $y$ —[кодирует принадлежность к]→ положительный класс
  таргет $y$ —[кодирует принадлежность к]→ отрицательный класс
  таргет $y$ —[принимает значения из]→ множество $\{-1, 1\}$
  линейная модель —[параметризуется]→ веса

In [19]:
save_json(metrics.summary(), out / 'benchmark_metrics.json')
print(f'Saved benchmark_metrics.json to {out}/')

Saved benchmark_metrics.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l02_qwen_7b/output/


## Benchmark — multi-method / multi-threshold sweep

In [20]:
is_multi = config.clustering.multi_method or config.clustering.is_multi_threshold

if not is_multi:
    print('Skipped: multi-method / multi-threshold not enabled in config')
    multi_metrics = None
else:
    total = sum(len(mr.graphs) for mr in multi.methods.values())
    print(f'Evaluating {total} configurations...')
    multi_metrics = evaluate_multi_graph(
        multi=multi,
        gt=gt_graph,
        source_text=source_text,
        embedder=embedder,
        tau_node=TAU_NODE,
        tau_edge=TAU_EDGE,
        beta=BETA,
        node_weight=NODE_WEIGHT,
        edge_weight=EDGE_WEIGHT,
        node_window=NODE_WINDOW,
        edge_window=EDGE_WINDOW,
    )
    print(f'Done: {sum(len(v) for v in multi_metrics.values())} variants evaluated')

Evaluating 23 configurations...


Batches: 100%|██████████| 1/1 [00:00<00:00, 130.78it/s]

Done: 23 variants evaluated


In [21]:
if multi_metrics:
    print_multi_metrics(multi_metrics, sort_by='graph_score')

method        param                  pred_n pred_e  matched_n  matched_e    P_n    R_n    F_n    P_e    R_e    F_e   graph
--------------------------------------------------------------------------------------------------------------------------
agglomerative 0.394                      55    111         55         51  0.696  0.696  0.696  0.348  0.757  0.477  0.6081
agglomerative 0.467                      33     89         33         51  0.750  0.450  0.563  0.427  0.745  0.543  0.5547
agglomerative 0.322                      79    135         55         51  0.506  0.726  0.596  0.292  0.773  0.424  0.5271
agglomerative 0.539                      15     49         15         49  0.831  0.227  0.356  0.746  0.717  0.731  0.5062
agglomerative 0.250                     197    241         55         51  0.209  0.749  0.327  0.167  0.788  0.275  0.3064
agglomerative 0.611                       3      5          3          5  0.889  0.049  0.092  0.895  0.088  0.160  0.1191
agglomerative 0.

In [22]:
if multi_metrics:
    method, param, best_m = best_variant(multi_metrics, by='graph_score')
    best_graph = multi.methods[method].graphs[param]
    print(f'Best variant: method={method}, param={param}')
    print(f'  graph: {len(best_graph.nodes)} nodes, {len(best_graph.edges)} edges')
    print()
    print_metrics(best_m)

Best variant: method=kmeans, param=k=55
  graph: 55 nodes, 105 edges

GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.6164

Nodes (pred=55, gt=55, matched=55, tau=0.6, beta=1):
  TP(soft)  = 38.3266
  precision = 0.6968
  recall    = 0.6968
  F1        = 0.6968
Edges (pred=105, gt=51, matched=51, tau=0.6, beta=1):
  TP(soft)  = 38.6658
  precision = 0.3682
  recall    = 0.7582
  F1        = 0.4957


In [23]:
if multi_metrics:
    save_json(multi_metrics_to_dict(multi_metrics), out / 'benchmark_metrics_multi.json')
    print(f'Saved benchmark_metrics_multi.json to {out}/')

Saved benchmark_metrics_multi.json to /home/platoon/graph/semantic-graph/exps/FINAL_EXPS/phase1b_llms/exp_l02_qwen_7b/output/
